In [ ]:
import torch
import numpy as np
from tqdm import tqdm

from utils import *
from src.iipw import *

d1 = 10000
d2 = 1000
r = 10
p = 0.1
ob = 10
sample = "iid"
if torch.cuda.is_available():
    device = 'cuda:1'
else:
    device = 'cpu'

# dataset
dataset = "syn"
#print(dataset)

total_t = 5
total_T_var_list = []
total_T_p_var_list = []
total_T_err_list = []
total_T_p_err_list = []
for t in range(total_t):
    M = load_normalized_data_syn(r, d1, d2, device)
    
    runs = 100
    T_p_list = []
    T_list = []
    T_p_err_list = []
    T_err_list = []
    for run in tqdm(range(runs)):

        if sample == "iid":
            # IID sample
            observed_M, masks = get_uniform_masks(M, p)
        else:
            # few entry sample
            observed_M, masks = get_random_samples_per_row(M.cpu().numpy(), ob)
            p = ob/d2
            observed_M = torch.from_numpy(observed_M).float().to(device)
            masks = torch.from_numpy(masks).to(device)
        # observed second-moment matrix
        MTM = M.T @ M
        normalized_MTM = MTM / d1
        second_moment_observe_M =  observed_M.T @ observed_M
        T_masks = 1 * (second_moment_observe_M!=0)

        T = iipw_T(observed_M)
        T_list.append(T.unsqueeze(0).cpu())
        T_err_list.append(compute_err_tensor(estimation_matrix=T, groundtruth_matrix=normalized_MTM, mask=T_masks).item())

        T_p = prob_T(observed_M, p)
        T_p_list.append(T_p.unsqueeze(0).cpu())
        T_p_err_list.append(compute_err_tensor(estimation_matrix=T_p, groundtruth_matrix=normalized_MTM, mask=T_masks).item())

    total_T_err_list.append(np.mean(T_err_list))
    total_T_p_err_list.append(np.mean(T_p_err_list))

    var_T = compute_var(T_list)
    var_T_p = compute_var(T_p_list)

    var_T_p_value = (var_T_p).mean()
    var_T_value = (var_T).mean()
 
    total_T_p_var_list.append(var_T_p_value.item())
    total_T_var_list.append(var_T_value.item())


print(f"mean of err of T_p: {np.mean(total_T_p_err_list)} +- {np.std(total_T_p_err_list)} ")
print(f"mean of err of T: {np.mean(total_T_err_list)} +- {np.std(total_T_err_list)} ")

print(f"mean of var of T_p: {np.mean(total_T_p_var_list)} +- {np.std(total_T_p_var_list)} ")
print(f"mean of var of T: {np.mean(total_T_var_list)} +- {np.std(total_T_var_list)} ")
